# Week 05 · 机器学习：数据划分与可靠评测

监督学习从特征 X 预测标签 y。分类预测离散类别，回归预测连续值。训练集用于拟合参数，验证集或交叉验证用于选择配置，测试集用于最后一次评估。提前用完整数据拟合标准化器会泄露测试分布；Pipeline 可以把预处理放进每个交叉验证折内部。

Accuracy=正确数/总数；Precision=TP/(TP+FP)，Recall=TP/(TP+FN)，F1 是 Precision 与 Recall 的调和平均。少数类重要时高准确率可能只是一直猜多数类。欠拟合表现为训练和验证都差；过拟合常表现为训练好但验证差。随机种子控制部分随机性，不保证跨硬件逐位一致。

下面使用 sklearn 自带乳腺癌数据（公开 Wisconsin Diagnostic Breast Cancer 数据集），比较 Dummy、逻辑回归和决策树。类 0=malignant、1=benign，因此如果关注恶性病例，Recall 的 pos_label 要设 0，不能机械使用默认正类。错误分析应看真实误判、特征分布和样本来源，不能用测试集反复调参。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

dataset = load_breast_cancer(as_frame=True)
X, y = dataset.data, dataset.target
assert X.isna().sum().sum() == 0
print("样本数、特征数：", X.shape, "类别比例：", y.value_counts(normalize=True).to_dict())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
models = {
    "majority baseline": DummyClassifier(strategy="most_frequent"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "tree": DecisionTreeClassifier(max_depth=4, random_state=42),
}
rows=[]
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="f1_macro")
    model.fit(X_train, y_train)
    prediction=model.predict(X_test)
    rows.append({"model":name,"cv_f1_macro":cv.mean(),"accuracy":accuracy_score(y_test,prediction),
                 "malignant_precision":precision_score(y_test,prediction,pos_label=0,zero_division=0),
                 "malignant_recall":recall_score(y_test,prediction,pos_label=0),
                 "f1_macro":f1_score(y_test,prediction,average="macro")})
results=pd.DataFrame(rows)
display(results)
prediction=models["logistic"].predict(X_test)
print("混淆矩阵，行=真实、列=预测：\n",confusion_matrix(y_test,prediction))
errors=X_test.loc[prediction!=y_test].copy()
errors["true"]=y_test.loc[errors.index]
errors["predicted"]=prediction[prediction!=y_test]
display(errors.head(5))
results.to_csv("week05-metrics.csv",index=False)
assert set(X_train.index).isdisjoint(X_test.index)

## 练习 / Exercises
把 max_depth 从 1 改成 None，仅在训练集交叉验证中选择；解释 precision 和 recall 的业务取舍。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
depth_scores={depth:cross_val_score(DecisionTreeClassifier(max_depth=depth,random_state=42),X_train,y_train,cv=5,scoring="f1_macro").mean() for depth in [1,2,4,8,None]}
print("只用训练集 CV 选择深度：",depth_scores)
print("最佳深度：",max(depth_scores,key=depth_scores.get))